In [ ]:
import os
import pickle
from functools import partial

import datasets
import flax.linen as nn
import jax
import jax.numpy as jnp
import numpy as np
import optax
import tensorflow_datasets as tfds
import tqdm
from flax.metrics import tensorboard
from jax_resnet import resnet
from matplotlib import pyplot as plt

print(f"JAX & XLA => {jax.default_backend()} / {jax.lib.xla_bridge.get_backend().platform}")

In [ ]:
# Define folders

# SHEAR2COSMO_dir = "/net/GECO/pisco/SHEAR2COSMO/"
SHEAR2COSMO_dir = "/home/rbrunet/RESS"

IMAGESIMS_dir = SHEAR2COSMO_dir + "/tensorflow_datasets"
LOGS_dir = SHEAR2COSMO_dir + "/logs"
MODELS_dir = SHEAR2COSMO_dir + "/MODELS"
PLOTS_dir = SHEAR2COSMO_dir + "/PLOTS"

In [ ]:
# Define hyperparameters and suffix
batch_size = 64

# Define a scheduler for the learning rate
# total_steps = 50_000
total_steps = 1000

lri = 0.002

lr2 = 0.3
step_2 = 30_000

lr_scheduler = optax.piecewise_constant_schedule(
    init_value=lri,
    boundaries_and_scales={
        int(step_2): lr2,
    },
)


# Define a simple MSE regression loss
def loss_fn(params, batch):
    return jnp.mean(jnp.sum((model.apply(params, batch["x"]) - batch["y"]) ** 2, axis=-1))


case = (
    "cosmoslics_resnet18_mse_lri"
    + str(lri)
    + "_lr2"
    + str(lr2)
    + "_batch"
    + str(batch_size)
    + "_steps"
    + str(total_steps)
)

In [ ]:
# Define our compressor
ResNet = partial(resnet.ResNet18, n_classes=4)


class NeuralCompressor(nn.Module):
    @nn.compact
    def __call__(self, x):
        return ResNet()(x)

In [ ]:
# Random seed for sampling from posterior
seed = jax.random.PRNGKey(0)

# Let's instantiate and initialize the model
model = NeuralCompressor()
params = model.init(jax.random.PRNGKey(42), jnp.zeros(shape=[1, 128, 128, 1]))

# Initializes an optimizer and optimizer state
optimizer = optax.adam(learning_rate=lr_scheduler)
opt_state = optimizer.init(params)

summary_writer = tensorboard.SummaryWriter(LOGS_dir + "/" + case)

In [ ]:
dset = tfds.load("cosmo_slics", split="train", data_dir=IMAGESIMS_dir)
# Let's collect a few examples to check their distributions
cutouts = []
specz = []
for batch, entry in enumerate(dset.take(100)):
    cutouts.append(entry["Map"])
cutouts = np.stack(cutouts)
cutouts[np.isnan(cutouts)] = 0

In [ ]:
for i, b in enumerate(["SNR"]):
    plt.hist(cutouts[...].flatten(), 100, label=b)
plt.legend()
plt.yscale("log")

In [ ]:
plt.figure(figsize=(15, 3))
for i, b in enumerate(["Map"]):
    plt.subplot(1, 5, i + 1)
    plt.imshow(cutouts[0, :, :], vmin=-4, vmax=8)
    plt.title(b)
    plt.axis("off")

plt.figure(figsize=(15, 3))
for i, b in enumerate(["Map"]):
    plt.subplot(1, 5, i + 1)
    plt.imshow(cutouts[1, :, :], vmin=-4, vmax=8)
    plt.title(b)
    plt.axis("off")

In [ ]:
# Let's evaluate the noise standard deviation in each band, and apply range
# compression accordingly
# from astropy.stats import mad_std
# scaling = []

# for i,b in enumerate(['i']):
#  plt.hist(cutouts[...].flatten(),100, label=b,alpha=0.5,);
#  sigma = mad_std(cutouts[...].flatten())
#  scaling.append(sigma)
#  plt.axvline(sigma, color='C%d'%i,alpha=0.5)
#  plt.axvline(-sigma, color='C%d'%i,alpha=0.5)
# plt.legend()

In [ ]:
# Let's have a look at this distribution if we rescale it between 0 and 1 using maxand min SNR value
# over the dataset and center it

min_Mapmap = -6.3
max_Mapmap = 21.63

for i, b in enumerate(["i"]):
    # plt.hist(cutouts[...].flatten()/scaling[i],100, label=b,alpha=0.5,
    plt.hist(
        (cutouts[...].flatten() - min_Mapmap) / (max_Mapmap - min_Mapmap),
        100,
        label=b,
        alpha=0.5,
        range=[0, 1],
    )
plt.legend()
plt.yscale("log")

In [ ]:
# a common approach for range compression is to apply arcsinh to suppress the
# high amplitude values
# for i,b in enumerate(['i']):
#  plt.hist(np.arcsinh(cutouts[...].flatten()/scaling[i]/3),100,
#           label=b, alpha=0.5,
#           range=[-5,5]);
# plt.legend()
# plt.yscale('log')

In [ ]:
# we can have a look at individual postage stamps with or without this scaling
# plt.subplot(121)
# plt.imshow(cutouts[0,:,:]/scaling[0])
# plt.title('Before')
# plt.subplot(122)
# plt.imshow(np.arcsinh(cutouts[0,:,:]/scaling[0]/3))
# plt.title('After');

In [ ]:
import tensorflow as tf


def preprocessing(example):
    def range_compression(img):
        # return tf.clip_by_value(
        #     tf.math.asinh(tf.expand_dims(img, axis=-1) / tf.constant(scaling) / 3.0), -10, 10
        # )
        return tf.clip_by_value(
            (tf.expand_dims(img, axis=-1) - min_Mapmap) / (max_Mapmap - min_Mapmap), -10, 10
        )

    # Our preprocessing function only returns the postage stamps, and the shear
    return range_compression(example["Map"]), example["CP"]

In [ ]:
# batch_size = 64

# Load the data
dset = tfds.load("cosmo_slics", split="train[:80%]", data_dir=IMAGESIMS_dir)
dset = dset.repeat()
dset = dset.shuffle(5000)
dset = dset.map(preprocessing)
dset = dset.batch(batch_size, drop_remainder=True)
dset = dset.prefetch(-1)
dset = dset.as_numpy_iterator()

In [ ]:
dset_val = tfds.load("cosmo_slics", split="train[80%:90%]", data_dir=IMAGESIMS_dir)
dset_val = dset_val.repeat()
dset_val = dset_val.map(preprocessing)
dset_val = dset_val.batch(batch_size, drop_remainder=True)
dset_val = dset_val.prefetch(-1)
dset_val = dset_val.as_numpy_iterator()

In [ ]:
@jax.jit
def update_model(params, opt_state, batch):
    # Computes the gradients of the model
    loss, grads = jax.value_and_grad(loss_fn)(params, batch)

    # Computes the weights updates and apply them
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss

In [ ]:
# os.environ["TENSORBOARD_BINARY"] = "/home/rbrunet/micromamba/envs/SHEAR2COSMO/bin/tensorboard"
os.environ["TENSORBOARD_BINARY"] = "/net/GECO/nas12c/users/nmartinet/deeppython2/bin/tensorboard"

%load_ext tensorboard
%tensorboard --logdir={LOGS_dir}    

In [ ]:
(batch[0].shape)

In [ ]:
for i in tqdm.tqdm(range(total_steps)):
    batch = next(dset)
    key, seed = jax.random.split(seed)
    batch = {"x": jnp.nan_to_num(batch[0]), "y": batch[1]}

    # Apply the update function
    params, opt_state, loss = update_model(params, opt_state, batch)

    # Periodically exporting the value of the loss function, every 10 steps
    if i % 20 == 0:
        summary_writer.scalar("train_loss", loss, i)
        summary_writer.scalar("learning_rate", lr_scheduler(i), i)

        batch = next(dset_val)
        batch = {"x": jnp.nan_to_num(batch[0]), "y": batch[1]}
        loss_val = loss_fn(params, batch)
        summary_writer.scalar("val_loss", loss_val, i)

    # Save model parameters
    if i % 1000 == 0:
        with open(MODELS_dir + "/" + case + "_temp" + str(i) + ".pckl", "wb") as handle:
            pickle.dump(params, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(MODELS_dir + "/" + case + "_complete.pckl", "wb") as handle:
    pickle.dump(params, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Read model parameters
with open(MODELS_dir + "/" + case + "_complete.pckl", "rb") as handle:
    params_trained = pickle.load(handle)

In [ ]:
dset_test = tfds.load("cosmo_slics", split="train[90%:]", data_dir=IMAGESIMS_dir)
dset_test = dset_test.map(preprocessing)
dset_test = dset_test.batch(batch_size, drop_remainder=True)
dset_test = dset_test.prefetch(-1)
dset_test = dset_test.as_numpy_iterator()

In [ ]:
preds = []
truth = []

# Applying the model
for batch in dset_test:
    # preds.append(model.apply(params, jnp.nan_to_num(batch[0])))
    preds.append(model.apply(params_trained, jnp.nan_to_num(batch[0])))
    truth.append(batch[1])

preds = np.concatenate(preds)
truth = np.concatenate(truth)

In [ ]:
poly_Om = np.polyfit(truth[:, 0], preds[:, 0], 1)
poly_S8 = np.polyfit(truth[:, 1], preds[:, 1], 1)
poly_h = np.polyfit(truth[:, 2], preds[:, 2], 1)
poly_w = np.polyfit(truth[:, 3], preds[:, 3], 1)

plt.figure(figsize=(11, 11))

#
plt.subplot(221)

t = np.linspace(0.0, 0.6)
plt.plot(t, t, color="k")
plt.fill_between(t, t - 0.05 * 0.3, t + 0.05 * 0.3, color="gray", alpha=0.5)
plt.plot(t, poly_Om[1] + poly_Om[0] * t, color="C1")

plt.scatter(truth[:, 0], preds[:, 0], alpha=0.2, label="$\Omega_{\\rm m}$", color="C1")
plt.xlabel("true")
plt.ylabel("pred")

plt.legend()
plt.title(
    "$\Omega_{\\rm m}^{\\rm pred} = %0.4f \\times \Omega_{\\rm m}^{\\rm true} + %0.4f$"
    % (poly_Om[0], poly_Om[1])
)
plt.ylim(0.0, 0.6)
plt.xlim(0.0, 0.6)

#
plt.subplot(222)

t = np.linspace(0.5, 1.0)
plt.plot(t, t, color="k")
plt.fill_between(t, t - 0.05 * 0.75, t + 0.05 * 0.75, color="gray", alpha=0.5)
plt.plot(t, poly_S8[1] + poly_S8[0] * t, color="C0")

plt.scatter(truth[:, 1], preds[:, 1], alpha=0.2, label="$S_8$", color="C0")
plt.xlabel("true")
plt.ylabel("pred")

plt.legend()
plt.title("$S_8^{\\rm pred} = %0.4f \\times S_8^{\\rm true} + %0.4f$" % (poly_S8[0], poly_S8[1]))
plt.ylim(0.5, 1.0)
plt.xlim(0.5, 1.0)

#
plt.subplot(223)

t = np.linspace(0.55, 0.85)
plt.plot(t, t, color="k")
plt.fill_between(t, t - 0.05 * 0.7, t + 0.05 * 0.7, color="gray", alpha=0.5)
plt.plot(t, poly_h[1] + poly_h[0] * t, color="C2")

plt.scatter(truth[:, 2], preds[:, 2], alpha=0.2, label="$h$", color="C2")
plt.xlabel("true")
plt.ylabel("pred")

plt.legend()
plt.title("$h^{\\rm pred} = %0.4f \\times h^{\\rm true} + %0.4f$" % (poly_h[0], poly_h[1]))
plt.ylim(0.55, 0.85)
plt.xlim(0.55, 0.85)

#
plt.subplot(224)

t = np.linspace(-2.5, 0.0)
plt.plot(t, t, color="k")
plt.fill_between(t, t - 0.05 * 1.2, t + 0.05 * 1.2, color="gray", alpha=0.5)
plt.plot(t, poly_w[1] + poly_w[0] * t, color="C3")

plt.scatter(truth[:, 3], preds[:, 3], alpha=0.2, label="$w$", color="C3")
plt.xlabel("true")
plt.ylabel("pred")

plt.legend()
plt.title("$w^{\\rm pred} = %0.4f \\times w^{\\rm true} + %0.4f$" % (poly_w[0], poly_w[1]))
plt.ylim(-2.5, 0.0)
plt.xlim(-2.5, 0.0)

#
plt.savefig(PLOTS_dir + "/" + case + "_predtrue.png")